In [0]:
-- Create or replace the table to store parsed PDF content
CREATE OR REPLACE TABLE gshen_catalog.ai_parse.enbridge_fact_sheet_pdf AS(
  SELECT
    path,  -- File path of the PDF
    ai_parse_document(content) parsed_content  -- Parsed content as a VARIANT using AI parsing function
  FROM READ_FILES('/Volumes/gshen_catalog/ai_parse/enbridge', format => 'binaryFile')  -- Read binary PDF files from the specified path
);

-- Preview the parsed PDF content
SELECT * FROM gshen_catalog.ai_parse.enbridge_fact_sheet_pdf;

-- Explode the parsed content into individual elements (e.g., pages or sections)
SELECT
  path,  -- File path
  exploded_elements.pos,  -- Position of the element in the array
  CAST(exploded_elements.value:page_id AS INT) AS page_id,  -- Extract and cast page_id from the element
  exploded_elements.value:content  -- Extract content from the element
FROM gshen_catalog.ai_parse.enbridge_fact_sheet_pdf as t,
LATERAL variant_explode_outer(t.parsed_content:document:elements) as exploded_elements;  -- Explode the elements array

-- Concatenate the content to create a single row per page and store in a new table
CREATE OR REPLACE TABLE gshen_catalog.ai_parse.enbridge_fact_sheet_chunked as (
  WITH q as( 
    SELECT
      path,  -- File path
      exploded_elements.pos,  -- Position of the element
      CAST(exploded_elements.value:page_id AS INT) AS page_id,  -- Page ID
      exploded_elements.value:content  -- Content of the element
    FROM gshen_catalog.ai_parse.enbridge_fact_sheet_pdf as t,
    LATERAL variant_explode_outer(t.parsed_content:document:elements) as exploded_elements  -- Explode elements
  ) 
  SELECT
    xxhash64(path, page_id) as pk,
    path,  -- File path
    page_id,  -- Page ID
    concat_ws(' ', collect_list(content)) AS combined_content  -- Concatenate all content for the same page into a single string
  FROM q
  GROUP BY path, page_id  -- Group by file and page
  ORDER BY path, page_id  -- Order by file and page
);

-- Preview the chunked table with combined content per page
SELECT * FROM  gshen_catalog.ai_parse.enbridge_fact_sheet_chunked;